# Train 3D-UNet Temperature Surrogate

Train a grid-size-invariant 3D-UNet surrogate for one-step temperature prediction. The baseline input uses three temperature history frames, current 3D velocity, and static/boundary forcing channels, then predicts the next 3D temperature field.

In [ ]:
from pathlib import Path
import os
import json
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from IPython.display import clear_output, display

from temperature_surrogate_3d import (
    Temperature3DCase,
    TemperatureSurrogateDataset,
    SurrogateStats,
    UNet3D,
    discover_case_dirs,
    evaluate,
    split_cases,
    train_one_epoch,
)

BUNDLE_ROOT = Path.cwd()
if not (BUNDLE_ROOT / 'temperature_surrogate_3d.py').exists():
    BUNDLE_ROOT = Path('velocity/temperature_field_bundle').resolve()
print('Bundle root:', BUNDLE_ROOT)

## Configuration

Use `PATCH_SIZE` for grid-size-invariant patch training. Keep each dimension divisible by `2 ** DEPTH`.

In [ ]:
DATA_ROOT = BUNDLE_ROOT / 'outputs' / 'temperature_3d_surrogate_dataset_48'
RUN_DIR = BUNDLE_ROOT / 'outputs' / 'temperature_3d_unet_surrogate'
RUN_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT_NAME = 'best_temperature_3d_unet_surrogate.pt'
best_path = RUN_DIR / BEST_CHECKPOINT_NAME

HISTORY_STEPS = 3
PATCH_SIZE = (8, 32, 32)  # (z, y, x). Increase only after the smoke training is stable.
DEPTH = 3
BASE_CHANNELS = 12
EPOCHS = 20
LEARNING_RATE = 2e-4
SAMPLES_PER_CASE_TRAIN = 96
SAMPLES_PER_CASE_VAL = 24
VAL_FRACTION = 0.2
NUM_WORKERS = min(4, max(1, (os.cpu_count() or 2) // 2))
VALIDATE_EVERY = 2
SEED = 7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 2 if (device.type == 'cuda' and torch.cuda.get_device_properties(0).total_memory >= 40 * 1024**3) else 1

torch.backends.cudnn.benchmark = (device.type == 'cuda')
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('Total GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('Batch size:', BATCH_SIZE)
print('Num workers:', NUM_WORKERS)
print('Validate every:', VALIDATE_EVERY, 'epoch(s)')
print('Data root:', DATA_ROOT)


## Load Cases


In [ ]:
import time

case_dirs = discover_case_dirs(DATA_ROOT)
if not case_dirs:
    raise FileNotFoundError(
        f'No 3D temperature cases found under {DATA_ROOT}. Run Generate_Temperature_3D_Surrogate_Dataset_48.ipynb first.'
    )

print(f'Discovered {len(case_dirs)} case directories under {DATA_ROOT}', flush=True)
start_time = time.time()
cases = []
for idx, path in enumerate(case_dirs, start=1):
    print(f'[load {idx}/{len(case_dirs)}] {path.name}', flush=True)
    cases.append(Temperature3DCase(path))
print(f'Loaded {len(cases)} cases in {time.time() - start_time:.1f} seconds', flush=True)

train_cases, val_cases = split_cases(cases, val_fraction=VAL_FRACTION, seed=SEED)
if not train_cases:
    train_cases = val_cases

stats_path = RUN_DIR / 'surrogate_stats.json'
train_case_dirs = [str(case.case_dir) for case in train_cases]
stats_payload = None
if stats_path.exists():
    try:
        stats_payload = json.loads(stats_path.read_text())
    except json.JSONDecodeError:
        stats_payload = None

if stats_payload and stats_payload.get('train_case_dirs') == train_case_dirs:
    print(f'Loading cached stats from {stats_path} (delete it to force recompute)', flush=True)
    stats = SurrogateStats(**stats_payload['stats'])
else:
    print('Computing surrogate normalization stats from training cases...', flush=True)
    stats_start = time.time()
    stats = SurrogateStats.from_cases(
        train_cases,
        max_values_per_array=20_000,
        progress=lambda message: print(message, flush=True),
    )
    stats_path.write_text(json.dumps({
        'stats': stats.__dict__,
        'train_case_dirs': train_case_dirs,
        'val_fraction': VAL_FRACTION,
        'seed': SEED,
    }, indent=2))
    print(f'Saved stats cache to {stats_path}', flush=True)
    print(f'Stats computed in {time.time() - stats_start:.1f} seconds', flush=True)

print(f'Cases: {len(cases)} | train: {len(train_cases)} | val: {len(val_cases)}')
print('First temperature shape [time,z,y,x]:', cases[0].temperature.shape)
print('First velocity shape [time,z,y,x]:', cases[0].u.shape)
print(stats)


In [ ]:
train_ds = TemperatureSurrogateDataset(
    train_cases,
    stats,
    history_steps=HISTORY_STEPS,
    patch_size=PATCH_SIZE,
    samples_per_case=SAMPLES_PER_CASE_TRAIN,
    random_crop=True,
    deterministic_sampling=False,
    seed=SEED,
)
val_ds = TemperatureSurrogateDataset(
    val_cases,
    stats,
    history_steps=HISTORY_STEPS,
    patch_size=PATCH_SIZE,
    samples_per_case=SAMPLES_PER_CASE_VAL,
    random_crop=True,
    deterministic_sampling=True,
    seed=SEED + 10_000,
)

loader_kwargs = {
    'num_workers': NUM_WORKERS,
    'pin_memory': (device.type == 'cuda'),
}
if NUM_WORKERS > 0:
    loader_kwargs['persistent_workers'] = True
    loader_kwargs['prefetch_factor'] = 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

batch = next(iter(train_loader))
print('Input batch:', tuple(batch['x'].shape))
print('Target batch:', tuple(batch['y'].shape))
print('Mask batch:', tuple(batch['mask'].shape))
print('Input channels:', train_ds.in_channels)


## Model

The model is fully convolutional. That is the part that preserves the grid-size-invariant behavior: train on patches, infer on larger volumes with the same grid spacing.

In [ ]:
model = UNet3D(
    in_channels=train_ds.in_channels,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    depth=DEPTH,
    upsample_mode='trilinear',
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params/1e6:.2f}M')

## Training With Live Curves

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_mae_c': [], 'val_mae_c': []}
best_val = float('inf')
best_path = RUN_DIR / 'best_temperature_3d_unet_surrogate.pt'

def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], label='train')
    axes[0].plot(epochs, history['val_loss'], label='val')
    axes[0].set_title('Masked MSE loss')
    axes[0].set_xlabel('epoch')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    axes[1].plot(epochs, history['train_mae_c'], label='train')
    axes[1].plot(epochs, history['val_mae_c'], label='val')
    axes[1].set_title('MAE (deg C)')
    axes[1].set_xlabel('epoch')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    display(fig)
    plt.close(fig)

last_val_metrics = None
for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, device, stats, amp=True)

    if epoch == 1 or epoch % VALIDATE_EVERY == 0 or epoch == EPOCHS:
        val_metrics = evaluate(model, val_loader, device, stats)
        last_val_metrics = val_metrics
        scheduler.step(val_metrics['loss'])
        if val_metrics['loss'] < best_val:
            best_val = val_metrics['loss']
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'stats': stats.__dict__,
                'config': {
                    'history_steps': HISTORY_STEPS,
                    'patch_size': PATCH_SIZE,
                    'depth': DEPTH,
                    'base_channels': BASE_CHANNELS,
                    'in_channels': train_ds.in_channels,
                    'batch_size': BATCH_SIZE,
                },
                'history': history,
                'epoch': epoch,
            }, best_path)
    else:
        val_metrics = last_val_metrics if last_val_metrics is not None else {'loss': float('nan'), 'mae_c': float('nan')}

    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['train_mae_c'].append(train_metrics['mae_c'])
    history['val_mae_c'].append(val_metrics['mae_c'])

    clear_output(wait=True)
    print(f'Epoch {epoch}/{EPOCHS}')
    print(f"train loss={train_metrics['loss']:.5f}, val loss={val_metrics['loss']:.5f}")
    print(f"train MAE={train_metrics['mae_c']:.3f} C, val MAE={val_metrics['mae_c']:.3f} C")
    print('best checkpoint:', best_path)
    print('validated this epoch:', epoch == 1 or epoch % VALIDATE_EVERY == 0 or epoch == EPOCHS)
    plot_history(history)

with open(RUN_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('Saved history to', RUN_DIR / 'training_history.json')


## Quick Visual Check

This plots one validation patch at the middle z-slice.

## Validation Bias Audit And Thermal Calibration



In [ ]:
import numpy as np
from collections import defaultdict

checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

def parse_case_tokens(case_name):
    parts = case_name.split('_')
    geometry = next((p for p in parts if p.startswith('g')), "g???")
    velocity = next((p for p in parts if p.startswith('v')), "v??")
    thermal = next((p for p in parts if p.startswith('t')), "t??")
    return geometry, velocity, thermal

def make_full_volume_input(case, stats, history_steps):
    helper = TemperatureSurrogateDataset(
        [case],
        stats,
        history_steps=history_steps,
        patch_size=None,
        samples_per_case=1,
        random_crop=False,
        deterministic_sampling=True,
        seed=SEED,
    )
    t = case.n_training_steps - 1
    crop = (slice(None), slice(None), slice(None))
    temp_history = np.asarray(case.temperature[t - history_steps + 1 : t + 1], dtype=np.float32)
    u = np.asarray(case.u[t], dtype=np.float32)
    v = np.asarray(case.v[t], dtype=np.float32)
    w = np.asarray(case.w[t], dtype=np.float32)
    speed = np.sqrt(u * u + v * v + w * w)
    channels = [helper._norm_temp(frame) for frame in temp_history]
    channels.extend([u / stats.velocity_scale, v / stats.velocity_scale, w / stats.velocity_scale])
    if helper.include_speed:
        channels.append(speed / stats.velocity_scale)
    channels.extend(helper._static_channels(case, crop, t))
    x = np.stack(channels, axis=0).astype(np.float32)
    y_true = np.asarray(case.temperature[t + 1], dtype=np.float32)
    mask = helper._fluid_mask(case, crop).astype(bool)
    return x, y_true, mask, t + 1

def sliding_starts(size, tile, overlap):
    if tile >= size:
        return [0]
    stride = max(1, tile - overlap)
    starts = list(range(0, size - tile + 1, stride))
    if starts[-1] != size - tile:
        starts.append(size - tile)
    return starts

def predict_full_volume(case, model, device, stats, history_steps, tile_size=PATCH_SIZE, overlap=(2, 8, 8)):
    x_full, y_true, mask, target_frame = make_full_volume_input(case, stats, history_steps)
    _, zdim, ydim, xdim = x_full.shape
    if tile_size is None:
        tile_size = (zdim, ydim, xdim)
    tz, ty, tx = tile_size
    oz, oy, ox = overlap
    pred_sum = np.zeros((zdim, ydim, xdim), dtype=np.float32)
    pred_count = np.zeros((zdim, ydim, xdim), dtype=np.float32)
    z_starts = sliding_starts(zdim, tz, oz)
    y_starts = sliding_starts(ydim, ty, oy)
    x_starts = sliding_starts(xdim, tx, ox)
    with torch.no_grad():
        for z0 in z_starts:
            for y0 in y_starts:
                for x0 in x_starts:
                    tile = x_full[:, z0:z0+tz, y0:y0+ty, x0:x0+tx]
                    xt = torch.from_numpy(tile[None, ...]).to(device, non_blocking=True)
                    with torch.amp.autocast('cuda', enabled=device.type == "cuda"):
                        pred_tile = model(xt)[0, 0]
                    pred_tile = (pred_tile.float().cpu().numpy() * stats.temp_std + stats.temp_mean).astype(np.float32)
                    pred_sum[z0:z0+tz, y0:y0+ty, x0:x0+tx] += pred_tile
                    pred_count[z0:z0+tz, y0:y0+ty, x0:x0+tx] += 1.0
    pred_c = pred_sum / np.maximum(pred_count, 1.0)
    return pred_c, y_true, mask, target_frame

val_records = []
thermal_bias_groups = defaultdict(list)

print(f'Running full-volume validation on {len(val_cases)} case(s)...', flush=True)
for idx, case in enumerate(val_cases, start=1):
    print(f'[eval {idx}/{len(val_cases)}] {case.case_dir.name}', flush=True)
    pred_c, true_c, mask, frame_id = predict_full_volume(
        case,
        model,
        device,
        stats,
        history_steps=HISTORY_STEPS,
        tile_size=PATCH_SIZE,
        overlap=(2, 8, 8),
    )
    diff = pred_c - true_c
    diff_masked = diff[mask]
    geometry, velocity, thermal = parse_case_tokens(case.case_dir.name)
    record = {
        'case_id': case.case_dir.name,
        'geometry': geometry,
        'velocity': velocity,
        'thermal': thermal,
        'frame_id': int(frame_id),
        'bias_c': float(np.mean(diff_masked)),
        'mae_c': float(np.mean(np.abs(diff_masked))),
        'rmse_c': float(np.sqrt(np.mean(diff_masked ** 2))),
    }
    val_records.append(record)
    thermal_bias_groups[thermal].append(record['bias_c'])

val_records = sorted(val_records, key=lambda item: item['mae_c'], reverse=True)
thermal_bias_correction = {key: float(np.mean(values)) for key, values in sorted(thermal_bias_groups.items())}

print()
print('Validation summary')
for record in val_records:
    print(
        f"{record['case_id']}: bias={record['bias_c']:+.3f} C, "
        f"MAE={record['mae_c']:.3f} C, RMSE={record['rmse_c']:.3f} C"
    )

all_bias = np.array([record['bias_c'] for record in val_records], dtype=np.float32)
all_mae = np.array([record['mae_c'] for record in val_records], dtype=np.float32)
all_rmse = np.array([record['rmse_c'] for record in val_records], dtype=np.float32)
print()
print('Overall averages')
print(f'mean bias: {all_bias.mean():+.3f} C')
print(f'MAE:       {all_mae.mean():.3f} C')
print(f'RMSE:      {all_rmse.mean():.3f} C')
print()
print('Grouped by thermal case')
for thermal, bias in thermal_bias_correction.items():
    thermal_mae = [record['mae_c'] for record in val_records if record['thermal'] == thermal]
    thermal_rmse = [record['rmse_c'] for record in val_records if record['thermal'] == thermal]
    print(f"{thermal} bias={bias:+.3f} C, MAE={np.mean(thermal_mae):.3f} C, RMSE={np.mean(thermal_rmse):.3f} C")

(RUN_DIR / 'thermal_bias_correction.json').write_text(json.dumps({
    'created_from_checkpoint': str(best_path),
    'history_steps': HISTORY_STEPS,
    'patch_size': PATCH_SIZE,
    'thermal_bias_correction_c': thermal_bias_correction,
    'validation_records': val_records,
}, indent=2))
print()
print(f"Saved thermal bias correction to {RUN_DIR / 'thermal_bias_correction.json'}")


## Real Geometry Validation Visualization


In [ ]:
from pathlib import Path

# Change these two lines when you want a different validation example or a different height.
VIS_CASE_NAME = val_records[0]['case_id']  # for example: "case_g180_v02_t01"
VIS_Z_INDEX = 6  # change this to inspect another z level

selected_case = next(case for case in val_cases if case.case_dir.name == VIS_CASE_NAME)
pred_c, true_c, mask, frame_id = predict_full_volume(
    selected_case,
    model,
    device,
    stats,
    history_steps=HISTORY_STEPS,
    tile_size=PATCH_SIZE,
    overlap=(2, 8, 8),
)

geometry, velocity, thermal = parse_case_tokens(selected_case.case_dir.name)
bias_correction = thermal_bias_correction.get(thermal, 0.0)
pred_c_corrected = pred_c - bias_correction

velocity_cache_dir = selected_case._velocity_cache_dir()
building_mask = np.load(velocity_cache_dir / 'building_mask_2d.npy').astype(bool)

VIS_Z_INDEX = max(0, min(int(VIS_Z_INDEX), true_c.shape[0] - 1))
valid_values = np.concatenate([true_c[mask], pred_c[mask], pred_c_corrected[mask]])
vmin = float(np.nanpercentile(valid_values, 2))
vmax = float(np.nanpercentile(valid_values, 98))
raw_err = pred_c - true_c
corrected_err = pred_c_corrected - true_c
err_lim = float(np.nanpercentile(np.abs(np.concatenate([raw_err[mask], corrected_err[mask]])), 98))

fig, axes = plt.subplots(1, 5, figsize=(24, 4.8), constrained_layout=True)
im0 = axes[0].imshow(true_c[VIS_Z_INDEX], cmap='YlOrRd', vmin=vmin, vmax=vmax, origin='upper')
axes[0].set_title(f'True T\n{selected_case.case_dir.name} | frame {frame_id} | z={VIS_Z_INDEX}')
im1 = axes[1].imshow(pred_c[VIS_Z_INDEX], cmap='YlOrRd', vmin=vmin, vmax=vmax, origin='upper')
axes[1].set_title('Predicted T (raw)')
im2 = axes[2].imshow(pred_c_corrected[VIS_Z_INDEX], cmap='YlOrRd', vmin=vmin, vmax=vmax, origin='upper')
axes[2].set_title(f'Predicted T (corrected)\nsubtract {bias_correction:+.3f} C for {thermal}')
im3 = axes[3].imshow(raw_err[VIS_Z_INDEX], cmap='coolwarm', vmin=-err_lim, vmax=err_lim, origin='upper')
axes[3].set_title(f'Raw error\nmean bias={raw_err[mask].mean():+.3f} C | MAE={np.abs(raw_err[mask]).mean():.3f} C')
im4 = axes[4].imshow(corrected_err[VIS_Z_INDEX], cmap='coolwarm', vmin=-err_lim, vmax=err_lim, origin='upper')
axes[4].set_title(f'Corrected error\nmean bias={corrected_err[mask].mean():+.3f} C | MAE={np.abs(corrected_err[mask]).mean():.3f} C')
for ax in axes:
    ax.contour(building_mask.astype(float), levels=[0.5], colors='black', linewidths=0.45, alpha=0.7)
    ax.axis('off')
fig.colorbar(im2, ax=axes[:3], shrink=0.85, label='deg C')
fig.colorbar(im4, ax=axes[3:], shrink=0.85, label='deg C')
plt.show()

print('How to change height: edit VIS_Z_INDEX near the top of this cell, then rerun the cell.')
print('How to change case: edit VIS_CASE_NAME near the top of this cell, then rerun the cell.')


In [ ]:
# Multi-case, two-height visualization without correction.
# All cases are rotated back to the same reference geometry orientation (g000).
# Outside the study area is shown in white.

import numpy as np
import matplotlib.pyplot as plt

CASE_NAMES = [
    val_records[0]['case_id'],
    val_records[min(1, len(val_records) - 1)]['case_id'],
    val_records[min(2, len(val_records) - 1)]['case_id'],
    val_records[min(3, len(val_records) - 1)]['case_id'],
]

Z_LEVELS = [2, 6]

temp_cmap = plt.get_cmap('YlOrRd').copy()
temp_cmap.set_bad('white')

err_cmap = plt.get_cmap('coolwarm').copy()
err_cmap.set_bad('white')


def rotation_k_from_geometry_tag(geometry_tag):
    mapping = {
        'g000': 0,
        'g090': 1,
        'g180': 2,
        'g270': 3,
    }
    return mapping.get(geometry_tag, 0)


def rotate_back_2d(field_2d, k):
    if k % 4 == 0:
        return field_2d
    return np.rot90(field_2d, -k)


def rotate_back_3d(field_3d, k):
    if k % 4 == 0:
        return field_3d
    return np.rot90(field_3d, -k, axes=(1, 2))


for case_name in CASE_NAMES:
    selected_case = next(case for case in val_cases if case.case_dir.name == case_name)

    pred_c, true_c, mask, frame_id = predict_full_volume(
        selected_case,
        model,
        device,
        stats,
        history_steps=HISTORY_STEPS,
        tile_size=PATCH_SIZE,
        overlap=(2, 8, 8),
    )

    raw_err = pred_c - true_c
    geometry, velocity, thermal = parse_case_tokens(selected_case.case_dir.name)
    rot_k = rotation_k_from_geometry_tag(geometry)

    velocity_cache_dir = selected_case._velocity_cache_dir()
    building_mask = np.load(velocity_cache_dir / 'building_mask_2d.npy').astype(bool)

    study_area_path = velocity_cache_dir / 'study_area_mask_2d.npy'
    if study_area_path.exists():
        study_area_mask_2d = np.load(study_area_path).astype(bool)
    else:
        study_area_mask_2d = np.any(mask, axis=0)

    true_c_vis = rotate_back_3d(true_c, rot_k)
    pred_c_vis = rotate_back_3d(pred_c, rot_k)
    raw_err_vis = rotate_back_3d(raw_err, rot_k)
    mask_vis = rotate_back_3d(mask.astype(bool), rot_k)
    building_mask_vis = rotate_back_2d(building_mask, rot_k)
    study_area_mask_vis = rotate_back_2d(study_area_mask_2d, rot_k)

    valid_z_levels = [max(0, min(int(z), true_c_vis.shape[0] - 1)) for z in Z_LEVELS]

    in_domain = study_area_mask_vis
    vmin = float(np.nanpercentile(true_c_vis[:, in_domain], 2))
    vmax = float(np.nanpercentile(true_c_vis[:, in_domain], 98))
    err_lim = float(np.nanpercentile(np.abs(raw_err_vis[:, in_domain]), 98))

    nrows = len(valid_z_levels)
    fig, axes = plt.subplots(
        nrows, 3,
        figsize=(15, 4.6 * nrows),
        constrained_layout=True
    )
    if nrows == 1:
        axes = axes[None, :]

    for row, z_idx in enumerate(valid_z_levels):
        true_slice = true_c_vis[z_idx].copy()
        pred_slice = pred_c_vis[z_idx].copy()
        err_slice = raw_err_vis[z_idx].copy()

        true_slice[~study_area_mask_vis] = np.nan
        pred_slice[~study_area_mask_vis] = np.nan
        err_slice[~study_area_mask_vis] = np.nan

        slice_mask = mask_vis[z_idx]
        slice_mae = float(np.mean(np.abs(raw_err_vis[z_idx][slice_mask]))) if np.any(slice_mask) else np.nan
        slice_bias = float(np.mean(raw_err_vis[z_idx][slice_mask])) if np.any(slice_mask) else np.nan

        im0 = axes[row, 0].imshow(
            true_slice, cmap=temp_cmap, vmin=vmin, vmax=vmax, origin='upper'
        )
        axes[row, 0].set_title(f"True T\n{case_name} | frame {frame_id} | z={z_idx}")

        im1 = axes[row, 1].imshow(
            pred_slice, cmap=temp_cmap, vmin=vmin, vmax=vmax, origin='upper'
        )
        axes[row, 1].set_title("Predicted T (raw)\nshown in reference geometry orientation")

        im2 = axes[row, 2].imshow(
            err_slice, cmap=err_cmap, vmin=-err_lim, vmax=err_lim, origin='upper'
        )
        axes[row, 2].set_title(f"Raw error\nbias={slice_bias:+.3f} C | MAE={slice_mae:.3f} C")

        for col in range(3):
            ax = axes[row, col]
            ax.contour(
                building_mask_vis.astype(float),
                levels=[0.5],
                colors='black',
                linewidths=0.45,
                alpha=0.7,
            )
            ax.set_xlabel('x index')
            ax.set_ylabel('y index')
            ax.tick_params(labelsize=8)
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(0.8)

    cbar1 = fig.colorbar(im1, ax=axes[:, :2], shrink=0.92, fraction=0.025, pad=0.02)
    cbar1.set_label('deg C')

    cbar2 = fig.colorbar(im2, ax=axes[:, 2], shrink=0.92, fraction=0.025, pad=0.02)
    cbar2.set_label('deg C')

    fig.suptitle(
        f"{case_name} | original tag: {geometry} | velocity: {velocity} | thermal: {thermal}",
        fontsize=13,
        y=1.02,
    )
    plt.show()

print("How to change cases: edit CASE_NAMES at the top of the cell.")
print("How to change heights: edit Z_LEVELS at the top of the cell.")